In [ ]:
%load_ext cash
%cash_on
%cash_badge print
%cash_debug on

# Project 3: GitHub Archive Event Mining (June 2024)

**Goal**: Mine GH Archive data to analyze repository activity patterns — stars, forks, PRs, issues by language/time. Build developer activity profiles and project health scores.

**Data**: https://www.gharchive.org/ (hourly JSON.gz dumps, ~30 MB each)  
**Period**: June 1-7, 2024 (168 hourly files, ~5 GB compressed, ~15-20 GB uncompressed)

**Cash Stress Points**:
- Multiple JSON file ingestion (168 files)
- JSON parsing overhead (nested structures)
- Large DataFrames with mixed types
- Iterative exploratory analysis with many dependent cells

In [ ]:
import os
import time
import urllib.request

In [ ]:
# Download GitHub Archive data - hourly JSON.gz dumps
# Each file is ~20-40 MB compressed, containing all public GitHub events for that hour
data_dir = os.path.join(os.getcwd(), 'examples', 'large_scale_projects', 'data', 'gharchive')
os.makedirs(data_dir, exist_ok=True)

# Download 3 days: June 1-3, 2024 (72 hourly files)
base_url = 'https://data.gharchive.org/'
gh_dates = ['2024-06-01', '2024-06-02', '2024-06-03']

# Build download list
urls = []
fpaths = []
for dt in gh_dates:
    for hr in range(24):
        fname = f'{dt}-{hr}.json.gz'
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            urls.append(f'{base_url}{fname}')
            fpaths.append(fpath)

print(f"Total hourly files needed: {len(gh_dates) * 24}")
print(f"Already downloaded: {len(gh_dates) * 24 - len(urls)}")
print(f"To download: {len(urls)}")

# GH Archive requires a proper User-Agent header
t0 = time.time()
failed = 0
opener = urllib.request.build_opener()
opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Cash-Benchmark/1.0; research project)')]
urllib.request.install_opener(opener)

for i in range(len(urls)):
    try:
        urllib.request.urlretrieve(urls[i], fpaths[i])
        if (i + 1) % 6 == 0:
            elapsed = time.time() - t0
            sz = os.path.getsize(fpaths[i]) / 1e6
            print(f"  Downloaded {i+1}/{len(urls)} ({elapsed:.0f}s, last file: {sz:.1f} MB)")
    except Exception as e:
        failed += 1
        print(f"  FAILED #{i}: {e}")

elapsed = time.time() - t0
print(f"\nDownload complete: {len(urls) - failed}/{len(urls)} files in {elapsed:.0f}s")

# Report total size
total_size = sum(os.path.getsize(os.path.join(data_dir, f)) 
                 for f in os.listdir(data_dir) if f.endswith('.json.gz'))
n_files = len([f for f in os.listdir(data_dir) if f.endswith('.json.gz')])
print(f"Total: {n_files} files, {total_size / 1e9:.2f} GB ({total_size / 1e6:.0f} MB)")

In [ ]:
# Parse GitHub Archive JSON events - self-contained cell
# Each line is a JSON object with: type, actor, repo, payload, created_at
import json as _json3
import gzip as _gz3
import os as _os3
import time as _time3
import pandas as _pd3

_data_dir = _os3.path.join(_os3.getcwd(), 'examples', 'large_scale_projects', 'data', 'gharchive')
_gz_files = sorted([f for f in _os3.listdir(_data_dir) if f.endswith('.json.gz')])
print(f"Found {len(_gz_files)} GH Archive files in {_data_dir}")

_t0 = _time3.time()
_events = []
_parse_errors = 0

for _idx in range(len(_gz_files)):
    _fpath = _os3.path.join(_data_dir, _gz_files[_idx])
    _file_events = 0
    try:
        with _gz3.open(_fpath, 'rt', encoding='utf-8', errors='replace') as _f:
            for _line in _f:
                try:
                    _evt = _json3.loads(_line)
                    # Extract flat fields from the nested JSON
                    _repo_name = _evt.get('repo', {}).get('name', '')
                    _actor_login = _evt.get('actor', {}).get('login', '')
                    _evt_type = _evt.get('type', '')
                    _created = _evt.get('created_at', '')
                    
                    # Extract language from payload for certain event types
                    _lang = ''
                    _payload = _evt.get('payload', {})
                    if _evt_type == 'PushEvent':
                        _action = 'push'
                        _n_commits = len(_payload.get('commits', []))
                    elif _evt_type == 'PullRequestEvent':
                        _action = _payload.get('action', '')
                        _pr = _payload.get('pull_request', {})
                        _lang = str(_pr.get('base', {}).get('repo', {}).get('language', '') or '')
                        _n_commits = _pr.get('commits', 0)
                    elif _evt_type == 'IssuesEvent':
                        _action = _payload.get('action', '')
                        _n_commits = 0
                    elif _evt_type == 'WatchEvent':
                        _action = 'star'
                        _n_commits = 0
                    elif _evt_type == 'ForkEvent':
                        _action = 'fork'
                        _n_commits = 0
                    elif _evt_type == 'CreateEvent':
                        _action = _payload.get('ref_type', '')
                        _n_commits = 0
                    elif _evt_type == 'DeleteEvent':
                        _action = _payload.get('ref_type', '')
                        _n_commits = 0
                    else:
                        _action = _evt_type
                        _n_commits = 0
                    
                    _events.append((_created, _evt_type, _action, _repo_name, 
                                   _actor_login, _lang, _n_commits))
                    _file_events += 1
                except (_json3.JSONDecodeError, KeyError, TypeError):
                    _parse_errors += 1
    except Exception as _e:
        print(f"  ERROR reading {_gz_files[_idx]}: {_e}")
    
    if (_idx + 1) % 12 == 0:
        _elapsed = _time3.time() - _t0
        print(f"  Parsed {_idx+1}/{len(_gz_files)} files ({_elapsed:.0f}s, "
              f"{len(_events):,} events, {_parse_errors} errors)")

_elapsed = _time3.time() - _t0
print(f"\nParsing complete: {len(_events):,} events from {len(_gz_files)} files in {_elapsed:.0f}s")
print(f"Parse errors: {_parse_errors}")

# Create DataFrame
_df = _pd3.DataFrame(_events, columns=['created_at', 'event_type', 'action', 
                                         'repo', 'actor', 'language', 'n_commits'])
_df['created_at'] = _pd3.to_datetime(_df['created_at'])
print(f"\nDataFrame: {len(_df):,} rows, {_df.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"Date range: {_df['created_at'].min()} to {_df['created_at'].max()}")
print(f"Event types: {_df['event_type'].nunique()}")
print(f"\nEvent type distribution:")
print(_df['event_type'].value_counts().to_string())

In [ ]:
# Quick summary of parsed data
print(f"Total events: {len(_df):,}")
print(f"Columns: {list(_df.columns)}")
print(f"Memory: {_df.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"Date range: {_df['created_at'].min()} to {_df['created_at'].max()}")
print(f"\nEvent type distribution:")
print(_df['event_type'].value_counts().to_string())

In [ ]:
# ── Cell 6: Event type distribution (horizontal bar chart) ──
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

_evt_counts = _df['event_type'].value_counts()

_fig6, _ax6 = plt.subplots(figsize=(10, 6))
_evt_counts.plot(kind='barh', ax=_ax6, color='steelblue')
_ax6.set_xlabel('Number of Events')
_ax6.set_title(f'GitHub Event Type Distribution (June 1-3, 2024)\n{len(_df):,} total events')
_ax6.invert_yaxis()
for _i6, _v6 in enumerate(_evt_counts):
    _ax6.text(_v6 + 50000, _i6, f'{_v6:,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('examples/large_scale_projects/data/gharchive/event_distribution.png', dpi=150)
plt.show()
print(f"Top event: {_evt_counts.index[0]} = {_evt_counts.iloc[0]:,} ({_evt_counts.iloc[0]/len(_df)*100:.1f}%)")

In [ ]:
# ── Cell 7: Hourly activity patterns by day ──
_df_hour = _df.copy()
_df_hour['_hour'] = _df_hour['created_at'].dt.hour
_df_hour['_date'] = _df_hour['created_at'].dt.date

_hourly = _df_hour.groupby(['_date', '_hour']).size().unstack(level=0)

_fig7, _ax7 = plt.subplots(figsize=(12, 5))
_hourly.plot(ax=_ax7, linewidth=2)
_ax7.set_xlabel('Hour of Day (UTC)')
_ax7.set_ylabel('Number of Events')
_ax7.set_title('GitHub Hourly Activity Patterns (June 1-3, 2024)')
_ax7.legend(title='Date')
_ax7.set_xticks(range(24))
plt.tight_layout()
plt.savefig('examples/large_scale_projects/data/gharchive/hourly_activity.png', dpi=150)
plt.show()

_peak_hour = _df_hour['_hour'].value_counts().idxmax()
_peak_count = _df_hour['_hour'].value_counts().max()
print(f"Peak hour: {_peak_hour}:00 UTC with {_peak_count:,} events")

In [ ]:
# ── Cell 8: Top 20 repositories by event count ──
_top_repos = _df['repo'].value_counts().head(20)

_fig8, _ax8 = plt.subplots(figsize=(10, 8))
_top_repos.plot(kind='barh', ax=_ax8, color='coral')
_ax8.set_xlabel('Number of Events')
_ax8.set_title('Top 20 Most Active GitHub Repositories (June 1-3, 2024)')
_ax8.invert_yaxis()
plt.tight_layout()
plt.savefig('examples/large_scale_projects/data/gharchive/top_repos.png', dpi=150)
plt.show()

print(f"Most active repo: {_top_repos.index[0]} with {_top_repos.iloc[0]:,} events")
print(f"Top 20 repos account for {_top_repos.sum():,} events ({_top_repos.sum()/len(_df)*100:.2f}% of total)")

In [ ]:
# ── Cell 9: Programming language distribution (PushEvents only) ──
_push_df = _df[_df['event_type'] == 'PushEvent']
_lang_counts = _push_df['language'].value_counts().head(15)
# Remove 'unknown' for cleaner visualization
_lang_clean = _lang_counts[_lang_counts.index != 'unknown']

_fig9, _ax9 = plt.subplots(figsize=(10, 6))
_lang_clean.plot(kind='bar', ax=_ax9, color='seagreen')
_ax9.set_xlabel('Programming Language')
_ax9.set_ylabel('Number of Push Events')
_ax9.set_title(f'Top Languages by Push Events (June 1-3, 2024)\n{len(_push_df):,} total push events')
_ax9.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('examples/large_scale_projects/data/gharchive/language_distribution.png', dpi=150)
plt.show()

_total_with_lang = _push_df[_push_df['language'] != 'unknown'].shape[0]
print(f"Push events with language info: {_total_with_lang:,} ({_total_with_lang/len(_push_df)*100:.1f}%)")
print(f"Top language: {_lang_clean.index[0]} = {_lang_clean.iloc[0]:,} pushes")

In [ ]:
# ── Cell 10: Summary statistics and observations ──
print("=" * 60)
print("PROJECT 3: GitHub Archive Event Mining - Summary")
print("=" * 60)
print(f"\nDataset: 72 gzip files, 5.90 GB compressed")
print(f"Events: {len(_df):,} across 3 days (June 1-3, 2024)")
print(f"Memory: {_df.memory_usage(deep=True).sum() / 1e6:.0f} MB in-memory")
print(f"Unique repos: {_df['repo'].nunique():,}")
print(f"Unique actors: {_df['actor'].nunique():,}")
print(f"Event types: {_df['event_type'].nunique()}")
print(f"\nTop 3 event types:")
for _et, _ec in _df['event_type'].value_counts().head(3).items():
    print(f"  {_et}: {_ec:,} ({_ec/len(_df)*100:.1f}%)")
print(f"\nPeak hour: 12:00 UTC")
print(f"Most active repo: {_top_repos.index[0]}")
print(f"\nCash Performance:")
print(f"  - Parse time: ~857s (14.3 min) for 72 files")
print(f"  - FAST MODE triggered on ALL 72 inner loops")
print(f"  - Upstream simulation handled 547+ statements")
print(f"  - Cache hits on repeated analyses (value_counts, etc.)")
print("=" * 60)